In [ ]:
# Install (usually already installed in Colab)
!pip install pandas numpy scikit-learn tensorflow

# Imports
import pandas as pd
import numpy as np
import re
import pickle

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [7]:
import pandas as pd
#Reading the dataset, skipping bad lines to handle parsing errors with the Python engine
df = pd.read_csv('/content/train.csv', encoding='latin1', on_bad_lines='skip', engine='python')

In [9]:
df.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [11]:
import re

# Keep required columns
df = df[['comment_text', 'toxic']].dropna()

# Clean text
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df['clean_text'] = df['comment_text'].apply(clean_text)

df.head()

,comment_text,toxic,clean_text
0,Explanation\nWhy the edits made under my usern...,0,explanation why the edits made under my userna...
1,D'aww! He matches this background colour I'm s...,0,d aww he matches this background colour i m se...
2,"Hey man, I'm really not trying to edit war. It...",0,hey man i m really not trying to edit war it s...
3,"""\nMore\nI can't make any real suggestions on ...",0,more i can t make any real suggestions on impr...
4,"You, sir, are my hero. Any chance you remember...",0,you sir are my hero any chance you remember wh...


In [13]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_words = 20000
max_len = 100

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(df['clean_text'])

X = tokenizer.texts_to_sequences(df['clean_text'])
X = pad_sequences(X, maxlen=max_len)

y = df['toxic']

print("Shape:", X.shape)

Shape: (51034, 100)


In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [22]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    LSTM(64),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [21]:
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/10
1021/1021 ━━━━━━━━━━━━━━━━━━━━ 106s 99ms/step - accuracy: 0.9390 - loss: 0.1812 - val_accuracy: 0.9537 - val_loss: 0.1336
Epoch 2/10
1021/1021 ━━━━━━━━━━━━━━━━━━━━ 144s 102ms/step - accuracy: 0.9642 - loss: 0.1002 - val_accuracy: 0.9553 - val_loss: 0.1290
Epoch 3/10
1021/1021 ━━━━━━━━━━━━━━━━━━━━ 138s 98ms/step - accuracy: 0.9762 - loss: 0.0650 - val_accuracy: 0.9542 - val_loss: 0.1448
Epoch 4/10
1021/1021 ━━━━━━━━━━━━━━━━━━━━ 100s 98ms/step - accuracy: 0.9854 - loss: 0.0406 - val_accuracy: 0.9458 - val_loss: 0.1836
Epoch 5/10
1021/1021 ━━━━━━━━━━━━━━━━━━━━ 102s 100ms/step - accuracy: 0.9898 - loss: 0.0285 - val_accuracy: 0.9505 - val_loss: 0.2692
Epoch 6/10
1021/1021 ━━━━━━━━━━━━━━━━━━━━ 143s 101ms/step - accuracy: 0.9926 - loss: 0.0199 - val_accuracy: 0.9446 - val_loss: 0.2772
Epoch 7/10
1021/1021 ━━━━━━━━━━━━━━━━━━━━ 105s 102ms/step - accuracy: 0.9949 - loss: 0.0139 - val_accuracy: 0.9429 - val_loss: 0.3093
Epoch 8/10
1021/1021 ━━━━━━━━━━━━━━━━━━━━ 105s 102ms/step - accur

In [24]:
from sklearn.metrics import classification_report
y_pred = (model.predict(X_test) > 0.5).astype("int32")

print(classification_report(y_test, y_pred))

319/319 ━━━━━━━━━━━━━━━━━━━━ 8s 26ms/step
              precision    recall  f1-score   support

           0       0.89      0.53      0.66      9140
           1       0.10      0.44      0.16      1067

    accuracy                           0.52     10207
   macro avg       0.49      0.48      0.41     10207
weighted avg       0.81      0.52      0.61     10207



In [28]:
import pickle
# Save model
model.save("toxicity_model.h5")

# Save tokenizer
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("Saved successfully!")

Saved successfully!


In [27]:
from google.colab import files

files.download("toxicity_model.h5")
files.download("tokenizer.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [29]:
def predict_toxicity(text):
    text = clean_text(text)
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=max_len)
    prediction = model.predict(padded)[0][0]
    return prediction

# Test
comment = "You are a horrible person!"
score = predict_toxicity(comment)

print("Score:", score)
print("Toxic" if score > 0.5 else "Non-Toxic")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 194ms/step
Score: 0.49973685
Non-Toxic


In [30]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 104.8 MB/s eta 0:00:00


In [31]:
!pip install pyngrok

In [ ]:
%%writefile app.py

import streamlit as st
import pickle
import numpy as np
import pandas as pd
import re

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load model & tokenizer
model = load_model("toxicity_model.h5")

with open("tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

max_len = 100

# Text Cleaning
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Prediction Function
def predict_toxicity(text):
    text = clean_text(text)
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=max_len)
    prediction = model.predict(padded)[0][0]
    return prediction

# UI
st.set_page_config(page_title="Toxicity Detector", layout="centered")

st.title("💬 Comment Toxicity Detection")
st.write("Detect whether a comment is toxic or not using Deep Learning")

# Text input
user_input = st.text_area("Enter a comment:")

if st.button("Predict"):
    if user_input.strip() != "":
        score = predict_toxicity(user_input)

        st.subheader("Prediction Score:")
        st.write(round(score, 3))

        if score > 0.5:
            st.error("🚨 Toxic Comment")
        else:
            st.success("✅ Non-Toxic Comment")
    else:
        st.warning("Please enter some text.")

# CSV Upload for Bulk Prediction
st.subheader("📁 Bulk Prediction")

uploaded_file = st.file_uploader("Upload CSV with 'comment_text' column", type=["csv"])

if uploaded_file:
    df = pd.read_csv(uploaded_file)

    if "comment_text" in df.columns:
        df["clean_text"] = df["comment_text"].apply(clean_text)
        seq = tokenizer.texts_to_sequences(df["clean_text"])
        padded = pad_sequences(seq, maxlen=max_len)

        predictions = model.predict(padded)
        df["toxicity_score"] = predictions
        df["label"] = df["toxicity_score"].apply(lambda x: "Toxic" if x > 0.5 else "Non-Toxic")

        st.write(df.head())

        csv = df.to_csv(index=False).encode("utf-8")
        st.download_button("Download Results", csv, "predictions.csv", "text/csv")
    else:
        st.error("CSV must contain 'comment_text' column")

In [49]:
!streamlit run /content/app.py &>/content/logs.txt & npx localtunnel --port 8501 & curl ipv4.icanhazip.com

34.83.177.2
